
# API-контракты для приложения совместного бюджета в поездках

Этот notebook содержит проект REST API-контрактов, составленный по ER-схеме приложения для ведения совместного бюджета в поездках.

## Что учтено из схемы

Сущности на диаграмме:

- `users`
- `trips`
- `trip_partisipants`
- `roles`
- `expenses`
- `expense_splits`
- `statuses`
- `currencies`
- `expense_mode`

## Допущения

В схеме есть несколько, вероятно, опечаток. В контрактах ниже я привожу **нормализованные имена полей**, а рядом фиксирую соответствие со схемой:

- `trip_partisipants` → `trip_participants`
- `end_data` → `end_date`
- `create_by_participant_id` → `created_by_participant_id`
- `expense_data` → `expense_date`
- `expense_side_id` → `expense_mode_id`

Если нужно, я могу на следующем шаге сделать вторую версию notebook уже в формате **OpenAPI 3.1** с полноценными схемами `components/schemas`.



# 1. Базовые соглашения API

## Base URL

```text
/api/v1
```

## Формат данных

- `Content-Type: application/json`
- даты и время — в формате ISO 8601, UTC  
  пример: `2025-01-15T12:30:00Z`

## Авторизация

Для защищённых методов используется bearer token:

```http
Authorization: Bearer <access_token>
```

## Общая структура успешного ответа

```json
{
  "data": {}
}
```

или

```json
{
  "data": [],
  "meta": {
    "page": 1,
    "limit": 20,
    "total": 100
  }
}
```

## Общая структура ошибки

```json
{
  "error": {
    "code": "VALIDATION_ERROR",
    "message": "Validation failed",
    "details": {
      "field": [
        "must not be empty"
      ]
    }
  }
}
```

## Коды ответов

- `200 OK` — успешное чтение/обновление
- `201 Created` — сущность создана
- `204 No Content` — сущность удалена
- `400 Bad Request` — ошибка запроса
- `401 Unauthorized` — не авторизован
- `403 Forbidden` — нет прав
- `404 Not Found` — не найдено
- `409 Conflict` — конфликт состояния
- `422 Unprocessable Entity` — ошибка валидации



# 2. Основные модели данных

## 2.1 User

```json
{
  "id": 1,
  "first_name": "Ivan",
  "last_name": "Petrov",
  "nickname": "vanya",
  "email": "ivan@example.com",
  "created_at": "2025-01-10T10:00:00Z"
}
```

> `password_hash` из БД наружу не отдаётся.

## 2.2 Trip

```json
{
  "id": 10,
  "status": {
    "id": 1,
    "title": "planned"
  },
  "currency": {
    "id": 1,
    "title": "EUR"
  },
  "title": "Spain 2025",
  "description": "Trip to Barcelona",
  "budget": 250000,
  "invite_code": "BARCA2025",
  "cover_image_url": "https://cdn.example.com/trips/10.jpg",
  "start_date": "2025-06-01T00:00:00Z",
  "end_date": "2025-06-10T00:00:00Z",
  "created_at": "2025-01-10T10:00:00Z",
  "finished_at": null
}
```

## 2.3 Trip Participant

```json
{
  "id": 100,
  "trip_id": 10,
  "user": {
    "id": 1,
    "first_name": "Ivan",
    "last_name": "Petrov",
    "nickname": "vanya"
  },
  "role": {
    "id": 1,
    "title": "owner"
  }
}
```

## 2.4 Expense

```json
{
  "id": 1000,
  "trip_id": 10,
  "created_by_participant_id": 100,
  "title": "Dinner",
  "expense_mode": {
    "id": 2,
    "title": "cash"
  },
  "amount": 8400,
  "description": "Seafood restaurant",
  "expense_date": "2025-06-03T20:30:00Z",
  "created_at": "2025-06-03T20:35:00Z",
  "splits": [
    {
      "id": 1,
      "participant_id": 100,
      "share_amount": 2800
    },
    {
      "id": 2,
      "participant_id": 101,
      "share_amount": 2800
    },
    {
      "id": 3,
      "participant_id": 102,
      "share_amount": 2800
    }
  ]
}
```

## 2.5 Справочники

### Status
```json
{ "id": 1, "title": "planned" }
```

### Currency
```json
{ "id": 1, "title": "EUR" }
```

### Role
```json
{ "id": 1, "title": "owner" }
```

### ExpenseMode
```json
{ "id": 1, "title": "card" }
```



# 3. Авторизация и профиль


In [ ]:

auth_contracts = {
  "POST /auth/register": {
    "description": "Регистрация пользователя",
    "request": {
      "first_name": "Ivan",
      "last_name": "Petrov",
      "nickname": "vanya",
      "email": "ivan@example.com",
      "password": "StrongPassword123"
    },
    "responses": {
      "201": {
        "data": {
          "user": {
            "id": 1,
            "first_name": "Ivan",
            "last_name": "Petrov",
            "nickname": "vanya",
            "email": "ivan@example.com"
          },
          "access_token": "<jwt>",
          "token_type": "Bearer"
        }
      }
    }
  },
  "POST /auth/login": {
    "description": "Вход пользователя",
    "request": {
      "email": "ivan@example.com",
      "password": "StrongPassword123"
    },
    "responses": {
      "200": {
        "data": {
          "user": {
            "id": 1,
            "first_name": "Ivan",
            "last_name": "Petrov",
            "nickname": "vanya",
            "email": "ivan@example.com"
          },
          "access_token": "<jwt>",
          "token_type": "Bearer"
        }
      }
    }
  },
  "GET /auth/me": {
    "description": "Получить текущего пользователя",
    "responses": {
      "200": {
        "data": {
          "id": 1,
          "first_name": "Ivan",
          "last_name": "Petrov",
          "nickname": "vanya",
          "email": "ivan@example.com"
        }
      }
    }
  }
}

auth_contracts



# 4. Справочники


In [ ]:

dictionary_contracts = {
  "GET /statuses": {
    "description": "Список статусов поездки",
    "responses": {
      "200": {
        "data": [
          {"id": 1, "title": "planned"},
          {"id": 2, "title": "active"},
          {"id": 3, "title": "finished"}
        ]
      }
    }
  },
  "GET /currencies": {
    "description": "Список валют",
    "responses": {
      "200": {
        "data": [
          {"id": 1, "title": "EUR"},
          {"id": 2, "title": "USD"},
          {"id": 3, "title": "RUB"}
        ]
      }
    }
  },
  "GET /roles": {
    "description": "Список ролей участника поездки",
    "responses": {
      "200": {
        "data": [
          {"id": 1, "title": "owner"},
          {"id": 2, "title": "editor"},
          {"id": 3, "title": "viewer"}
        ]
      }
    }
  },
  "GET /expense-modes": {
    "description": "Список способов оплаты расхода",
    "responses": {
      "200": {
        "data": [
          {"id": 1, "title": "card"},
          {"id": 2, "title": "cash"}
        ]
      }
    }
  }
}

dictionary_contracts



# 5. Поездки


In [ ]:

trip_contracts = {
  "POST /trips": {
    "description": "Создать поездку",
    "request": {
      "title": "Spain 2025",
      "description": "Trip to Barcelona",
      "budget": 250000,
      "currency_id": 1,
      "cover_image_url": "https://cdn.example.com/trips/10.jpg",
      "start_date": "2025-06-01T00:00:00Z",
      "end_date": "2025-06-10T00:00:00Z"
    },
    "responses": {
      "201": {
        "data": {
          "id": 10,
          "status": {"id": 1, "title": "planned"},
          "currency": {"id": 1, "title": "EUR"},
          "title": "Spain 2025",
          "description": "Trip to Barcelona",
          "budget": 250000,
          "invite_code": "BARCA2025",
          "cover_image_url": "https://cdn.example.com/trips/10.jpg",
          "start_date": "2025-06-01T00:00:00Z",
          "end_date": "2025-06-10T00:00:00Z",
          "created_at": "2025-01-10T10:00:00Z",
          "finished_at": None
        }
      }
    }
  },
  "GET /trips": {
    "description": "Список поездок текущего пользователя",
    "query": {
      "status_id": 1,
      "page": 1,
      "limit": 20
    },
    "responses": {
      "200": {
        "data": [
          {
            "id": 10,
            "title": "Spain 2025",
            "budget": 250000,
            "currency": {"id": 1, "title": "EUR"},
            "status": {"id": 1, "title": "planned"},
            "start_date": "2025-06-01T00:00:00Z",
            "end_date": "2025-06-10T00:00:00Z"
          }
        ],
        "meta": {
          "page": 1,
          "limit": 20,
          "total": 1
        }
      }
    }
  },
  "GET /trips/{tripId}": {
    "description": "Детали поездки",
    "responses": {
      "200": {
        "data": {
          "id": 10,
          "title": "Spain 2025",
          "description": "Trip to Barcelona",
          "budget": 250000,
          "currency": {"id": 1, "title": "EUR"},
          "status": {"id": 1, "title": "planned"},
          "invite_code": "BARCA2025",
          "cover_image_url": "https://cdn.example.com/trips/10.jpg",
          "start_date": "2025-06-01T00:00:00Z",
          "end_date": "2025-06-10T00:00:00Z",
          "created_at": "2025-01-10T10:00:00Z",
          "finished_at": None,
          "participants_count": 3,
          "expenses_total": 8400
        }
      }
    }
  },
  "PATCH /trips/{tripId}": {
    "description": "Обновить поездку",
    "request": {
      "title": "Spain Summer 2025",
      "description": "Updated description",
      "budget": 300000,
      "currency_id": 2,
      "cover_image_url": "https://cdn.example.com/trips/10-new.jpg",
      "start_date": "2025-06-02T00:00:00Z",
      "end_date": "2025-06-11T00:00:00Z",
      "status_id": 2
    },
    "responses": {
      "200": {
        "data": {
          "id": 10,
          "title": "Spain Summer 2025",
          "description": "Updated description",
          "budget": 300000,
          "currency": {"id": 2, "title": "USD"},
          "status": {"id": 2, "title": "active"}
        }
      }
    }
  },
  "POST /trips/{tripId}/finish": {
    "description": "Завершить поездку",
    "responses": {
      "200": {
        "data": {
          "id": 10,
          "status": {"id": 3, "title": "finished"},
          "finished_at": "2025-06-11T08:00:00Z"
        }
      }
    }
  },
  "DELETE /trips/{tripId}": {
    "description": "Удалить поездку",
    "responses": {
      "204": {}
    }
  }
}

trip_contracts



# 6. Участники поездки


In [ ]:

participant_contracts = {
  "GET /trips/{tripId}/participants": {
    "description": "Список участников поездки",
    "responses": {
      "200": {
        "data": [
          {
            "id": 100,
            "trip_id": 10,
            "user": {
              "id": 1,
              "first_name": "Ivan",
              "last_name": "Petrov",
              "nickname": "vanya"
            },
            "role": {
              "id": 1,
              "title": "owner"
            }
          }
        ]
      }
    }
  },
  "POST /trips/join": {
    "description": "Вступить в поездку по invite_code",
    "request": {
      "invite_code": "BARCA2025"
    },
    "responses": {
      "201": {
        "data": {
          "participant_id": 101,
          "trip_id": 10,
          "role": {
            "id": 3,
            "title": "viewer"
          }
        }
      }
    }
  },
  "POST /trips/{tripId}/participants": {
    "description": "Добавить участника в поездку вручную",
    "request": {
      "user_id": 2,
      "role_id": 2
    },
    "responses": {
      "201": {
        "data": {
          "id": 101,
          "trip_id": 10,
          "user": {
            "id": 2,
            "first_name": "Anna",
            "last_name": "Sidorova",
            "nickname": "anya"
          },
          "role": {
            "id": 2,
            "title": "editor"
          }
        }
      }
    }
  },
  "PATCH /trips/{tripId}/participants/{participantId}": {
    "description": "Изменить роль участника",
    "request": {
      "role_id": 2
    },
    "responses": {
      "200": {
        "data": {
          "id": 101,
          "trip_id": 10,
          "role": {
            "id": 2,
            "title": "editor"
          }
        }
      }
    }
  },
  "DELETE /trips/{tripId}/participants/{participantId}": {
    "description": "Удалить участника из поездки",
    "responses": {
      "204": {}
    }
  }
}

participant_contracts



# 7. Расходы


In [ ]:

expense_contracts = {
  "POST /trips/{tripId}/expenses": {
    "description": "Создать расход",
    "request": {
      "created_by_participant_id": 100,
      "title": "Dinner",
      "expense_mode_id": 2,
      "amount": 8400,
      "description": "Seafood restaurant",
      "expense_date": "2025-06-03T20:30:00Z",
      "splits": [
        {
          "participant_id": 100,
          "share_amount": 2800
        },
        {
          "participant_id": 101,
          "share_amount": 2800
        },
        {
          "participant_id": 102,
          "share_amount": 2800
        }
      ]
    },
    "responses": {
      "201": {
        "data": {
          "id": 1000,
          "trip_id": 10,
          "created_by_participant_id": 100,
          "title": "Dinner",
          "expense_mode": {
            "id": 2,
            "title": "cash"
          },
          "amount": 8400,
          "description": "Seafood restaurant",
          "expense_date": "2025-06-03T20:30:00Z",
          "created_at": "2025-06-03T20:35:00Z",
          "splits": [
            {"id": 1, "participant_id": 100, "share_amount": 2800},
            {"id": 2, "participant_id": 101, "share_amount": 2800},
            {"id": 3, "participant_id": 102, "share_amount": 2800}
          ]
        }
      }
    }
  },
  "GET /trips/{tripId}/expenses": {
    "description": "Список расходов по поездке",
    "query": {
      "page": 1,
      "limit": 20,
      "created_by_participant_id": 100,
      "expense_mode_id": 2,
      "date_from": "2025-06-01T00:00:00Z",
      "date_to": "2025-06-10T23:59:59Z"
    },
    "responses": {
      "200": {
        "data": [
          {
            "id": 1000,
            "title": "Dinner",
            "amount": 8400,
            "expense_mode": {"id": 2, "title": "cash"},
            "expense_date": "2025-06-03T20:30:00Z",
            "created_by_participant_id": 100
          }
        ],
        "meta": {
          "page": 1,
          "limit": 20,
          "total": 1
        }
      }
    }
  },
  "GET /trips/{tripId}/expenses/{expenseId}": {
    "description": "Детали расхода",
    "responses": {
      "200": {
        "data": {
          "id": 1000,
          "trip_id": 10,
          "created_by_participant_id": 100,
          "title": "Dinner",
          "expense_mode": {"id": 2, "title": "cash"},
          "amount": 8400,
          "description": "Seafood restaurant",
          "expense_date": "2025-06-03T20:30:00Z",
          "created_at": "2025-06-03T20:35:00Z",
          "splits": [
            {"id": 1, "participant_id": 100, "share_amount": 2800},
            {"id": 2, "participant_id": 101, "share_amount": 2800},
            {"id": 3, "participant_id": 102, "share_amount": 2800}
          ]
        }
      }
    }
  },
  "PATCH /trips/{tripId}/expenses/{expenseId}": {
    "description": "Обновить расход",
    "request": {
      "title": "Dinner at beach",
      "expense_mode_id": 1,
      "amount": 9000,
      "description": "Updated description",
      "expense_date": "2025-06-03T21:00:00Z",
      "splits": [
        {
          "participant_id": 100,
          "share_amount": 3000
        },
        {
          "participant_id": 101,
          "share_amount": 3000
        },
        {
          "participant_id": 102,
          "share_amount": 3000
        }
      ]
    },
    "responses": {
      "200": {
        "data": {
          "id": 1000,
          "title": "Dinner at beach",
          "amount": 9000,
          "expense_mode": {
            "id": 1,
            "title": "card"
          }
        }
      }
    }
  },
  "DELETE /trips/{tripId}/expenses/{expenseId}": {
    "description": "Удалить расход",
    "responses": {
      "204": {}
    }
  }
}

expense_contracts



# 8. Балансы и аналитика

Эти методы не читаются напрямую из схемы, но почти всегда нужны такому приложению. Их можно считать рекомендованным слоем API поверх таблиц `expenses`, `expense_splits` и `trip_participants`.


In [ ]:

balance_contracts = {
  "GET /trips/{tripId}/balances": {
    "description": "Итоговые балансы участников по поездке",
    "responses": {
      "200": {
        "data": {
          "currency": {
            "id": 1,
            "title": "EUR"
          },
          "participants": [
            {
              "participant_id": 100,
              "user_id": 1,
              "full_name": "Ivan Petrov",
              "paid_amount": 8400,
              "owed_amount": 2800,
              "balance": 5600
            },
            {
              "participant_id": 101,
              "user_id": 2,
              "full_name": "Anna Sidorova",
              "paid_amount": 0,
              "owed_amount": 2800,
              "balance": -2800
            }
          ]
        }
      }
    }
  },
  "GET /trips/{tripId}/settlements": {
    "description": "Предлагаемые взаиморасчёты между участниками",
    "responses": {
      "200": {
        "data": [
          {
            "from_participant_id": 101,
            "to_participant_id": 100,
            "amount": 2800
          }
        ]
      }
    }
  },
  "GET /trips/{tripId}/summary": {
    "description": "Сводка по поездке",
    "responses": {
      "200": {
        "data": {
          "trip_id": 10,
          "budget": 250000,
          "total_expenses": 8400,
          "expenses_count": 1,
          "participants_count": 3,
          "budget_left": 241600
        }
      }
    }
  }
}

balance_contracts



# 9. Правила валидации

## Пользователь
- `first_name` — обязательное, 1..100 символов
- `last_name` — обязательное, 1..100 символов
- `nickname` — optional, до 100 символов, уникальность по бизнес-правилу желательно
- `email` — обязательное, уникальное, валидный email
- `password` — обязательное, минимум 8 символов

## Поездка
- `title` — обязательное, 1..255 символов
- `description` — optional
- `budget` — optional, `>= 0`
- `currency_id` — обязательное, должен существовать в `currencies`
- `start_date` — optional
- `end_date` — optional
- если заданы обе даты, то `end_date >= start_date`

## Участник поездки
- `trip_id` должен существовать
- `user_id` должен существовать
- `role_id` должен существовать
- комбинация `(trip_id, user_id)` должна быть уникальной

## Расход
- `trip_id` должен существовать
- `created_by_participant_id` должен существовать и принадлежать этой же поездке
- `title` — обязательное, 1..255 символов
- `expense_mode_id` — обязательное, должен существовать в `expense_mode`
- `amount` — обязательное, `> 0`
- `expense_date` — обязательное
- `splits` — минимум 1 элемент
- сумма `splits[].share_amount` должна быть равна `amount`
- каждый `participant_id` в `splits` должен принадлежать этой же поездке



# 10. Рекомендации по правам доступа

## Роли
В схеме есть таблица `roles`, но без явной матрицы прав. Для API разумно заложить такую модель:

- `owner`
  - создаёт/редактирует/удаляет поездку
  - добавляет/удаляет участников
  - меняет роли
  - может завершать поездку
- `editor`
  - читает поездку
  - создаёт/редактирует/удаляет расходы
  - видит балансы
- `viewer`
  - читает поездку
  - видит список расходов и балансы
  - без права редактирования

## Ограничения
- только участник поездки может видеть её данные
- только `owner` может менять состав участников и завершать поездку
- редактировать и удалять расход может:
  - его создатель, или
  - `owner`



# 11. Минимальный OpenAPI skeleton

Ниже не полный OpenAPI, а заготовка структуры, в которую можно перенести контракты выше.


In [ ]:

openapi_skeleton = {
  "openapi": "3.1.0",
  "info": {
    "title": "Trip Budget API",
    "version": "1.0.0"
  },
  "servers": [
    {"url": "/api/v1"}
  ],
  "paths": {
    "/auth/register": {},
    "/auth/login": {},
    "/auth/me": {},
    "/statuses": {},
    "/currencies": {},
    "/roles": {},
    "/expense-modes": {},
    "/trips": {},
    "/trips/{tripId}": {},
    "/trips/{tripId}/participants": {},
    "/trips/join": {},
    "/trips/{tripId}/expenses": {},
    "/trips/{tripId}/expenses/{expenseId}": {},
    "/trips/{tripId}/balances": {},
    "/trips/{tripId}/settlements": {},
    "/trips/{tripId}/summary": {}
  },
  "components": {
    "schemas": {
      "User": {},
      "Trip": {},
      "TripParticipant": {},
      "Expense": {},
      "ExpenseSplit": {},
      "Error": {}
    }
  }
}

openapi_skeleton



# 12. Итого

По этой схеме получаются такие основные bounded contexts API:

1. авторизация и профиль
2. справочники
3. поездки
4. участники поездки
5. расходы и сплиты
6. балансы и взаиморасчёты

Этот notebook уже можно использовать как основу для:
- backend-спецификации,
- swagger/openapi документа,
- frontend integration contract,
- Postman collection.

Следующий логичный шаг — преобразовать это в:
- **OpenAPI 3.1 YAML/JSON**
- **Postman collection**
- **SQL миграции / Prisma schema / TypeORM entities**
- **DTO для NestJS / FastAPI / Spring Boot**
